## Mounting Google Drive Data

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os

WORKING_DIRECTORY = '/content/drive/My Drive/Leaf_RCNN'

os.chdir(WORKING_DIRECTORY)

## Training Loop, Dataset Class, and DataLoaders

In [ ]:
import os, pandas as pd, torch, torchvision
from torchvision import transforms
import torchvision.transforms.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision.models.detection import FasterRCNN_ResNet50_FPN_Weights
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.models.detection import fasterrcnn_resnet50_fpn
import random

device = (
    torch.device("mps")  if torch.backends.mps.is_available()
    else torch.device("cuda") if torch.cuda.is_available()
    else torch.device("cpu")
)
print("Training on", device)

IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD  = (0.229, 0.224, 0.225)

import os, pandas as pd, torch, torchvision
import torchvision.transforms.functional as F
from torch.utils.data import Dataset

IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD  = (0.229, 0.224, 0.225)

#CsvDetectionDataset
class LeafDamageDetectionDataset(Dataset):
    def __init__(self, csv_path, img_root="resized", scale=0.15):
        self.df       = pd.read_csv(csv_path)
        self.img_root = img_root
        self.scale    = scale

        unique_labels          = self.df["category"].unique()
        self.label2id          = {lbl: i+1 for i, lbl in enumerate(unique_labels)}
        self.id2label          = {v: k for k, v in self.label2id.items()}  # optional

        print("Label mapping:", self.label2id)   # sanity-check

        self.groups = self.df.groupby("filename", sort=False)
        self.files  = list(self.groups.groups.keys())

    def __len__(self):
        return len(self.files) * 3

    def __getitem__(self, idx):
        N        = len(self.files)
        variant  = idx // N
        file_idx = idx %  N

        fn    = self.files[file_idx]
        rows  = self.groups.get_group(fn)

        path  = f"{self.img_root}/{fn}"
        img   = torchvision.io.read_image(path).float() / 255.0

        H, W  = img.shape[-2:]
        H2, W2 = int(H * self.scale), int(W * self.scale)
        new_H, new_W = int(H * self.scale), int(W * self.scale)
        img   = F.resize(img, (new_H, new_W))

        boxes = torch.as_tensor(
            rows[["xmin", "ymin", "xmax", "ymax"]].values,
            dtype=torch.float32
        ) * self.scale

        labels = torch.as_tensor(
            rows["category"].map(self.label2id).values,
            dtype=torch.int64
        )

        if variant == 1:
            img = F.hflip(img)
            boxes[:, [0, 2]] = W2 - boxes[:, [2, 0]]
        elif variant == 2:
            img = F.vflip(img)
            boxes[:, [1, 3]] = H2 - boxes[:, [3, 1]]


        img = F.normalize(img, IMAGENET_MEAN, IMAGENET_STD)

        target = {"boxes": boxes, "labels": labels}
        return img, target

def collate(batch):
    imgs, targets = list(zip(*batch))
    return list(imgs), list(targets)

def train():
    dataset = LeafDamageDetectionDataset(
        csv_path="../../data/rcnn_cropped_annotations_train.csv",
        scale=0.5
    )

    print("Creating DataLoader...")
    loader = DataLoader(dataset, batch_size=4, shuffle=True,
                        collate_fn=collate)
    print("Done")


    model = fasterrcnn_resnet50_fpn(weights="DEFAULT")
    num_classes = len(dataset.label2id) + 1
    in_feat = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = FastRCNNPredictor(in_feat, num_classes)
    model.to(device)

    optimizer = torch.optim.SGD(
        [p for p in model.parameters() if p.requires_grad],
        lr=1e-3, momentum=0.9, weight_decay=1e-4
    )

    for epoch in range(30):
        print(epoch)
        model.train()
        epoch_loss = 0.0
        for imgs, targets in loader:
            imgs    = [img.to(device) for img in imgs]
            targets = [{k: v.to(device) for k, v in t.items()} for t in targets]


            loss_dict = model(imgs, targets)
            loss = sum(loss_dict.values())

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()

        torch.save(model.state_dict(), "rcnn_weights2.pth")
        print(f"Epoch {epoch+1:02}  mean loss: {epoch_loss / len(loader):.4f}")

Training on cuda


## Training the Network

A powerful GPU is recommended. It can work with the A100s available in Google Colab.

In [ ]:
train()

Label mapping: {'Fungus': 1, 'Leaf': 2, 'Other Insect': 3, 'Mechanical Damage': 4, 'Leaf Miner': 5}
Creating DataLoader...
Done
0
Epoch 01  mean loss: 0.5816
1
Epoch 02  mean loss: 0.4367
2
Epoch 03  mean loss: 0.3900
3
Epoch 04  mean loss: 0.3488
4
Epoch 05  mean loss: 0.3170
5
Epoch 06  mean loss: 0.2956
6
Epoch 07  mean loss: 0.2682
7
Epoch 08  mean loss: 0.2465
8
Epoch 09  mean loss: 0.2348
9
Epoch 10  mean loss: 0.2176
10
Epoch 11  mean loss: 0.2053
11
Epoch 12  mean loss: 0.1910
12
Epoch 13  mean loss: 0.1791
13
Epoch 14  mean loss: 0.1715
14
Epoch 15  mean loss: 0.1656
15
Epoch 16  mean loss: 0.1579
16
Epoch 17  mean loss: 0.1528
17
Epoch 18  mean loss: 0.1444
18
Epoch 19  mean loss: 0.1365
19
Epoch 20  mean loss: 0.1304
20
Epoch 21  mean loss: 0.1275
21
Epoch 22  mean loss: 0.1223
22
Epoch 23  mean loss: 0.1197
23
Epoch 24  mean loss: 0.1177
24
Epoch 25  mean loss: 0.1129
25
Epoch 26  mean loss: 0.1099
26
Epoch 27  mean loss: 0.1079
27
Epoch 28  mean loss: 0.1006
28
Epoch 29  m

## Evaluating the Model

In [ ]:
from pathlib import Path
import torch, torchvision, pandas as pd
import torchvision.transforms.functional as F
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from tqdm.auto import tqdm
from collections import defaultdict
import numpy as np
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

TEST_CSV      = "../../data/rcnn_cropped_annotations_test.csv"
IMG_ROOT      = Path("resized")
CKPT_PATH     = "rcnn_weights.pth"
SCORE_THRESH  = 0.50
IOU_THRESH    = 0.50
SCALE_FACTOR  = 0.50
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD  = (0.229, 0.224, 0.225)

device = (
    torch.device("mps")  if torch.backends.mps.is_available()
    else torch.device("cuda") if torch.cuda.is_available()
    else torch.device("cpu")
)

def iou(box_a, box_b):
    """IoU between two [xmin, ymin, xmax, ymax] boxes (tensor or ndarray)."""
    ax1, ay1, ax2, ay2 = box_a
    bx1, by1, bx2, by2 = box_b
    inter_x1, inter_y1 = max(ax1, bx1), max(ay1, by1)
    inter_x2, inter_y2 = min(ax2, bx2), min(ay2, by2)
    inter_w, inter_h   = max(0, inter_x2 - inter_x1), max(0, inter_y2 - inter_y1)
    inter_area = inter_w * inter_h
    if inter_area == 0:
        return 0.0
    a_area = (ax2 - ax1) * (ay2 - ay1)
    b_area = (bx2 - bx1) * (by2 - by1)
    return inter_area / (a_area + b_area - inter_area)

def preprocess(img_path):
    img = torchvision.io.read_image(str(img_path)).float() / 255.0
    H, W = img.shape[-2:]
    img  = F.resize(img, (int(H * SCALE_FACTOR), int(W * SCALE_FACTOR)))
    img  = F.normalize(img, IMAGENET_MEAN, IMAGENET_STD)
    return img

ckpt       = torch.load(CKPT_PATH)
label_map  = {'Fungus': 1, 'Leaf': 2, 'Other Insect': 3, 'Mechanical Damage': 4, 'Leaf Miner': 5}
id2label   = {v: k for k, v in label_map.items()}
num_classes = len(label_map) + 1                  # + background

model = torchvision.models.detection.fasterrcnn_resnet50_fpn(weights=None)
in_feat = model.roi_heads.box_predictor.cls_score.in_features
model.roi_heads.box_predictor = FastRCNNPredictor(in_feat, num_classes)
model.load_state_dict(ckpt)
model.to(device).eval()

df   = pd.read_csv(TEST_CSV)
df[["xmin","ymin","xmax","ymax"]] *= SCALE_FACTOR

# group rows by filename
groups = df.groupby("filename", sort=False)

TP     = defaultdict(int)
FP     = defaultdict(int)
FN     = defaultdict(int)
iou_sum= defaultdict(float)
conf_pairs = []

for fname, rows in tqdm(groups, desc="Evaluating"):
    img_path = IMG_ROOT / fname
    net_img  = preprocess(img_path)

    with torch.inference_mode():
        pred  = model([net_img.to(device)])[0]
    keep = pred["scores"] >= SCORE_THRESH
    p_boxes  = pred["boxes"][keep].cpu().numpy()
    p_labels = pred["labels"][keep].cpu().numpy()

    gt_boxes  = rows[["xmin","ymin","xmax","ymax"]].values
    gt_labels = rows["category"].map(label_map).values
    matched_gt = np.zeros(len(gt_boxes), dtype=bool)
    matched_pred = np.zeros(len(p_boxes), dtype=bool)

    for pi, (pb, plab) in enumerate(zip(p_boxes, p_labels)):
        best_iou, best_j = 0., -1
        for gi, (gb, glab) in enumerate(zip(gt_boxes, gt_labels)):
            if matched_gt[gi] or plab != glab:      # class must match
                continue
            iou_val = iou(pb, gb)
            if iou_val > best_iou:
                best_iou, best_j = iou_val, gi
        if best_iou >= IOU_THRESH:
            matched_pred[pi] = True
            matched_gt[best_j] = True
            cls = plab
            TP[cls]   += 1
            iou_sum[cls] += best_iou
            conf_pairs.append((cls, cls))
        else:
            FP[plab] += 1

    for gi, matched in enumerate(matched_gt):
        if not matched:
            FN[gt_labels[gi]] += 1
            conf_pairs.append((gt_labels[gi], 0))

print("\nPer-class metrics (IoU ≥ %.2f, score ≥ %.2f)\n" % (IOU_THRESH, SCORE_THRESH))
header = "{:<15} {:>6} {:>6} {:>6} {:>8} {:>8} {:>8} {:>8}"
print(header.format("Class", "TP", "FP", "FN", "Prec", "Recall", "Acc", "mIoU"))

for cls_id, name in id2label.items():
    tp, fp, fn = TP[cls_id], FP[cls_id], FN[cls_id]
    prec = tp / (tp + fp) if tp + fp else 0.0
    rec  = tp / (tp + fn) if tp + fn else 0.0
    miou = iou_sum[cls_id] / tp if tp else 0.0
    acc  = tp / (tp + fp + fn) if tp + fp + fn else 0.0

    print(header.format(name, tp, fp, fn,
                        f"{prec:.3f}", f"{rec:.3f}", f"{acc:.3f}", f"{miou:.3f}"))

Evaluating:   0%|          | 0/48 [00:00<?, ?it/s]


Per-class metrics (IoU ≥ 0.50, score ≥ 0.50)

Class               TP     FP     FN     Prec   Recall      Acc     mIoU
Fungus              60     20     38    0.750    0.612    0.508    0.700
Leaf                48      0      0    1.000    1.000    1.000    0.987
Other Insect        31     38     17    0.449    0.646    0.360    0.671
Mechanical Damage      0      0      2    0.000    0.000    0.000    0.000
Leaf Miner          12      9      3    0.571    0.800    0.500    0.740


'\n# ─────────────────────────────────────────────────────────── CONFUSION MAT\n# build confusion matrix including background (id 0)\ny_true = [p[0] for p in conf_pairs]\ny_pred = [p[1] for p in conf_pairs]\nlabels_sorted = sorted(label_map.values())          # 0 first\nprint(id2label.keys())\ndisplay_names = [id2label[i] for i in range(1,6)]\ncm = confusion_matrix(y_true, y_pred, labels=labels_sorted)\ndisp = ConfusionMatrixDisplay(cm, display_labels=display_names)\nfig, ax = plt.subplots(figsize=(6, 6))\ndisp.plot(ax=ax, cmap="Blues", xticks_rotation=45, colorbar=False)\nax.set_title("Confusion / Precision matrix")\nplt.tight_layout()\nplt.show()\n'

## Saving the bounding boxes predictions

In [ ]:
from pathlib import Path
import pandas as pd
import torch, torchvision
import torchvision.transforms.functional as F
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from tqdm.auto import tqdm

TEST_CSV      = "cropped_annotations_test.csv"
IMG_ROOT      = Path("resized")
OUT_DIR       = Path("resized_pred")
CKPT_PATH     = "rcnn_weights.pth"
SCALE_FACTOR  = 0.50
SCORE_THRESH  = 0.50
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD  = (0.229, 0.224, 0.225)

device = (
    torch.device("mps")  if torch.backends.mps.is_available()
    else torch.device("cuda") if torch.cuda.is_available()
    else torch.device("cpu")
)
OUT_DIR.mkdir(exist_ok=True, parents=True)

# ───────────────────────────────────────── LOAD MODEL
ckpt       = torch.load(CKPT_PATH)
label_map  = {'Fungus': 1, 'Leaf': 2, 'Other Insect': 3, 'Mechanical Damage': 4, 'Leaf Miner': 5}
id2label   = {v: k for k, v in label_map.items()}
num_classes = len(label_map) + 1                  # + background

model = torchvision.models.detection.fasterrcnn_resnet50_fpn(weights=None)
in_feat = model.roi_heads.box_predictor.cls_score.in_features
model.roi_heads.box_predictor = FastRCNNPredictor(in_feat, num_classes)
model.load_state_dict(ckpt)
model.to(device).eval()

df = pd.read_csv(TEST_CSV)
filenames = df["filename"].unique()

def preprocess(img_path):
    img = torchvision.io.read_image(str(img_path)).float() / 255.0
    img = F.resize(img, [int(img.shape[-2]*SCALE_FACTOR),
                         int(img.shape[-1]*SCALE_FACTOR)])
    img = F.normalize(img, IMAGENET_MEAN, IMAGENET_STD)
    return img

def save_resized(img_tensor, out_path):
    torchvision.utils.save_image(img_tensor, str(out_path))

rows_scaled, rows_orig = [], []

for fname in tqdm(filenames, desc="Predicting"):
    src_path  = IMG_ROOT / fname
    dst_path  = OUT_DIR  / fname

    tensor = preprocess(src_path)
    save_resized(tensor, dst_path)

    H_r, W_r = tensor.shape[-2:]
    scale_back = 1.0 / SCALE_FACTOR

    with torch.inference_mode():
        pred = model([tensor.to(device)])[0]

    keep = pred["scores"] >= SCORE_THRESH
    boxes  = pred["boxes"][keep].cpu()
    labels = pred["labels"][keep].cpu()
    scores = pred["scores"][keep].cpu()

    for b, lab, sc in zip(boxes, labels, scores):
        x1,y1,x2,y2 = b.tolist()

        rows_scaled.append(
            dict(filename=fname,
                 xmin=x1, ymin=y1, xmax=x2, ymax=y2,
                 category=id2label[lab.item()],
                 score=sc.item())
        )

        rows_orig.append(
            dict(filename=fname,
                 xmin=x1*scale_back, ymin=y1*scale_back,
                 xmax=x2*scale_back, ymax=y2*scale_back,
                 category=id2label[lab.item()],
                 score=sc.item())
        )

pd.DataFrame(rows_scaled).to_csv("../../data/rcnn_predictions_scaled.csv",  index=False)
pd.DataFrame(rows_orig ).to_csv("../../data/rcnn_predictions_orig.csv",     index=False)

print("\n✓  Saved:")
print(f"   • Resized images    → {OUT_DIR}/")
print("   • CSV (scaled)       → ../../data/rcnn_predictions_scaled.csv")
print("   • CSV (orig coords)  → ../../data/rcnn_predictions_orig.csv")

Predicting:   0%|          | 0/48 [00:00<?, ?it/s]


✓  Saved:
   • Resized images    → resized_pred/
   • CSV (scaled)       → predictions_scaled.csv
   • CSV (orig coords)  → predictions_orig.csv
